In [ ]:

import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import pandas as pd
# from upsetplot import from_memberships, UpSet
import matplotlib.pyplot as plt

import math
import plotly.express as px

In [ ]:
df = pd.read_csv("../data/papers.csv")
total_papers = len(df)
df.columns

In [ ]:
bias_freqs_lst = df["LGBTQ+ Bias"].tolist()

bias_freqs = defaultdict(int)

for s in bias_freqs_lst:
    if isinstance(s, float):
        continue
    s = str(s).strip().lower()
    s = s.replace("_", " ")
    if s == "no":
        continue
    match = re.search(r"\((.*?)\)", s)
    if not match:
        continue
    items = match.group(1)
    for label in items.split(","):
        lab = label.strip()
        if lab:
            bias_freqs[lab] += 1

bias_freqs

In [ ]:
bias_freqs.pop("no", None)

bias_freqs["transgender"] += bias_freqs.pop("transgender man", 0)
bias_freqs["transgender"] += bias_freqs.pop("transgender woman", 0)

bias_freqs["gay or lesbian"] += bias_freqs.pop("gay", 0)
bias_freqs["gay or lesbian"] += bias_freqs.pop("lesbian", 0)
bias_freqs["gay or lesbian"] += bias_freqs.pop("homosexual", 0)

bias_freqs["heterosexual"] += bias_freqs.pop("and straight", 0)

bias_freqs["non-binary"] += bias_freqs.pop("nonbinary", 0)
bias_freqs["non-binary"] += bias_freqs.pop("gender nonconforming", 0)

bias_freqs.pop("lgbt+", 0)
bias_freqs

In [ ]:
def smart_title(s: str) -> str:
    return " ".join(w.capitalize() for w in s.split())

label_rotation = 0
x_tick_labelsize = 18
SHOW_AXIS_TITLES = True
axis_title_fontsize = 22
count_label_pad = 0.4
count_label_fontsize = 16
SAVE_SVG = True
svg_path = "lgbtq_bias.svg"
ylabel = "LGBTQ+ identity"
freq_var = bias_freqs

sorted_freqs = dict(sorted(freq_var.items(), key=lambda kv: kv[1], reverse=True))
groups = list(sorted_freqs.keys())
counts = list(sorted_freqs.values())
plot_df = pd.DataFrame({"group": [smart_title(g) for g in groups], "count": counts})

n_cats = len(groups)
fig, ax = plt.subplots(figsize=(12, max(8, 0.38 * n_cats)))

sns.barplot(
    data=plot_df,
    y="group",
    x="count",
    ax=ax,
    color="royalblue",
    edgecolor="black",
    linewidth=1,
    order=plot_df["group"],
)
sns.despine(ax=ax)

xmax_plot = 10
x_step = 2
x_gap = count_label_pad

for i, count in enumerate(counts):
    ax.text(count + x_gap, i, f"{count}", ha="left", va="center", fontsize=count_label_fontsize)

ax.set_xlim(0, xmax_plot)
ax.set_xticks(range(0, xmax_plot + 1, x_step))

if SHOW_AXIS_TITLES:
    ax.set_xlabel("Frequency", fontsize=axis_title_fontsize)
    ax.set_ylabel(ylabel, fontsize=axis_title_fontsize, labelpad=4)

ax.tick_params(axis="x", labelsize=x_tick_labelsize)
ax.tick_params(axis="y", labelsize=x_tick_labelsize, pad=2)
ax.set_yticklabels(ax.get_yticklabels(), rotation=label_rotation, ha="right")

longest = max((len(str(t.get_text())) for t in ax.get_yticklabels()), default=10)
fig.subplots_adjust(
    left=min(0.38, max(0.11, 0.085 + 0.0058 * longest)),
    right=0.98,
    top=0.98,
    bottom=0.08,
)
if SAVE_SVG:
    fig.savefig(svg_path, format="svg", bbox_inches="tight")
fig.savefig("lgbtq_bias.png", dpi=200, bbox_inches="tight")
plt.show()